In [ ]:
import pandas as pd

df = pd.read_csv('uncleaned_data.csv')

df.columns

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import pandas as pd


# 2. Clean column names a bit
df.columns = (
    df.columns
      .str.strip()
      .str.replace("\n", " ", regex=False)
)

df.columns


Index(['Date of Surgery', 'Sex', 'Age', 'Case/Type of Surgery', 'Perc screws?',
       'Open', 'Open Check V2', 'Standalone XLIF Check',
       'Retroperitoneal Approach (LLIF ± ALIF)',
       'Anterior + Posterior Apporoach', 'Osteotomies (yes/no)',
       'osteotomy level', 'T12-L1', 'L1-L2', 'L2-L3', 'L3-L4', 'L4-L5',
       'L5-S1', 'ALIF Count', 'Lateral Count', 'ACR (y=1)', 'ACR level',
       'Additional Procedures w/in surgery',
       'Pre op Diagnosis (back pain, adjacent segment disease, spondy)', 'BMI',
       'prior back surgeries? (y=1)', 'Average PI', 'VALIDATION COLUMN',
       'PI-LL angle mismatch', 'ABS PI-LL angle mismatch',
       'PI-LL Mismatch Category (1 = mismatch > +/- 9',
       'PI-LL Mismatch Category (1 = mismatch > +/- 10', 'PI  >50 (1 = PI>50)',
       'Post-op SS', 'Post-op SS Grouping', 'Roussouly Class', 'post PI',
       'post PT', 'post LL', 'post SVA', 'post-op date', 'most recent PI',
       'most recent PT', 'most recent LL', 'most recent SVA',


In [ ]:
target = "Time Until ASD Diagnosis (months)"

# Keep only rows where the target is known (patients who actually developed ASD)
asd_df = df[df[target].notna()].copy()
asd_df.shape
feature_cols = [
    # Demographics
    "Sex",
    "Age",
    "BMI",
    "prior back surgeries? (y=1)",

    # Surgical approach / technique
    "Perc screws?",
    "Open",
    "Open Check V2",
    "Standalone XLIF Check",
    "Retroperitoneal Approach (LLIF ± ALIF)",
    "Anterior + Posterior Apporoach",
    "Osteotomies (yes/no)",
    "ALIF Count",
    "Lateral Count",
    "ACR (y=1)",
    
    # Spinopelvic parameters
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "PI-LL Mismatch Category (1 = mismatch > +/- 9",
    "PI-LL Mismatch Category (1 = mismatch > +/- 10",
    "PI  >50 (1 = PI>50)",          # after newline cleanup
    "Post-op SS",
    "Post-op SS Grouping",
    "Roussouly Class",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",

    # Implant / hardware
    "titanium v PEEK (T=0,P=1)",

    # Hospital course
    "length of hospital stay (d)",

    # Binary complications (may carry signal for future ASD / revision)
    "post op complications? (Y/N)",
    "infection 1=yes",
    "DVT  1=yes",
    "PE  1=yes",
    "MI 1=yes",
    "femoral palsy (knee extension weakness) 1=yes",
    "hip flexion weakness (iliopsoas weakness)  1=yes",
    "transient weakness (post op clinic note)  1=yes",
    "pain/effort limited (stated)  1=yes",
    "acute thigh paresthesia (immediate post op)",
    "transient paresthesia (post op clinic note)",
    "psoas hematoma",
    "abdominal hernia (post op clinic note)",

    # Follow-up / other outcomes
    "follow up length (months)",
    "ASD B4 Surgery",
    "REVERIFIED ASD",
    "Time Without_ASD (months)",
    "need revision surgery? (Y=1)",
    "time since index surgery to revision surgery (m)",
]



In [ ]:
from pycaret.regression import setup, compare_models, pull, tune_model, finalize_model, save_model

# 1. Subset to features + target
data = asd_df[feature_cols + [target]].copy()

# 2. Optional: quick view
data.head()


,Sex,Age,BMI,prior back surgeries? (y=1),Perc screws?,Open,Open Check V2,Standalone XLIF Check,Retroperitoneal Approach (LLIF ± ALIF),Anterior + Posterior Apporoach,...,transient paresthesia (post op clinic note),psoas hematoma,abdominal hernia (post op clinic note),follow up length (months),ASD B4 Surgery,REVERIFIED ASD,Time Without_ASD (months),need revision surgery? (Y=1),time since index surgery to revision surgery (m),Time Until ASD Diagnosis (months)
4,M,67,28,0.0,0,0,0,1,1,0,...,0.0,0.0,NaN,16,0,1.0,6,0.0,NaN,6.0
11,M,40,45.8,1.0,1,0,0,0,0,1,...,0.0,0.0,NaN,106,0,1.0,78,0.0,NaN,78.0
15,M,67,26.2,1.0,1,0,0,0,0,1,...,0.0,0.0,NaN,50,0,1.0,3,0.0,NaN,3.0
30,F,71,27.6,1.0,0,0,0,1,1,0,...,0.0,0.0,NaN,84,1,1.0,27,1.0,41,27.0
35,F,77,31.79,0.0,0,0,0,1,1,0,...,0.0,0.0,NaN,12,0,1.0,12,0.0,NaN,13.0


In [ ]:
reg_setup = setup(
    data=data,
    target=target,
    session_id=42,
    train_size=0.8,                  # in your signature ✅

    # imputations (both are in your signature)
    imputation_type="simple",
    numeric_imputation="median",
    categorical_imputation="mode",

    # basic preprocessing
    preprocess=True,
    normalize=True,                  # ✅ in your signature
    normalize_method="zscore",

    # multicollinearity handling
    remove_multicollinearity=True,   # ✅ in your signature
    multicollinearity_threshold=0.9,

    # outliers (optional)
    remove_outliers=False,

    # cross-val settings
    fold_strategy="kfold",
    fold=10,
    fold_shuffle=True,

    # misc
    data_split_shuffle=True,
    verbose=True                     # or False if you want less logging
)


,Description,Value
0,Session id,42
1,Target,Time Until ASD Diagnosis (months)
2,Target type,Regression
3,Original data shape,"(125, 49)"
4,Transformed data shape,"(125, 84)"
5,Transformed train set shape,"(100, 84)"
6,Transformed test set shape,"(25, 84)"
7,Numeric features,35
8,Categorical features,13
9,Rows with missing values,100.0%


In [ ]:
# Compare a bunch of regression models
best_model = compare_models()
results = pull()
results.head()
print(best_model)


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,5.4880,148.8365,9.9060,0.3287,0.6521,1.0217,0.0590
gbr,Gradient Boosting Regressor,4.3741,131.5426,8.8266,0.2365,0.5650,0.8050,0.0390
et,Extra Trees Regressor,5.6566,158.8194,10.1823,0.1253,0.7616,1.4870,0.0620
par,Passive Aggressive Regressor,6.7483,146.7565,10.2268,0.0988,0.7728,1.4123,0.0380
ada,AdaBoost Regressor,6.4400,158.2625,10.1630,0.0378,0.8233,1.7213,0.0430
en,Elastic Net,8.6391,197.3346,12.6071,0.0018,0.9622,2.1277,0.0300
br,Bayesian Ridge,7.9683,168.7153,11.5888,-0.0070,0.9041,1.8540,0.0350
ridge,Ridge Regression,7.3352,156.8652,10.8984,-0.0230,0.8325,1.6162,0.0320
huber,Huber Regressor,7.6773,165.9063,11.5775,-0.0888,0.8632,1.8093,0.0330
llar,Lasso Least Angle Regression,7.3967,172.5364,11.3691,-0.1042,0.8706,1.8411,0.0320


RandomForestRegressor(n_jobs=-1, random_state=42)


In [ ]:
tuned_best = tune_model(best_model)
tuned_results = pull()
tuned_results.head()


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,4.4470,33.7410,5.8087,0.6132,0.7072,1.3432
1,2.8655,13.9239,3.7315,0.9475,0.5015,0.8279
2,5.8614,127.2447,11.2803,0.6375,0.6301,0.4404
3,2.7950,12.2873,3.5053,0.8806,0.4559,0.7295
4,14.1265,766.4213,27.6843,0.3323,0.8357,0.9110
5,3.1828,44.1049,6.6412,0.3958,0.7166,0.9943
6,2.6411,12.2912,3.5059,0.3448,0.7599,0.6383
7,9.3475,207.2045,14.3946,0.6439,1.1071,3.1248
8,8.2759,242.3145,15.5665,-2.0673,0.8720,1.9381


Fitting 10 folds for each of 10 candidates, totalling 100 fits


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,4.4470,33.7410,5.8087,0.6132,0.7072,1.3432
1,2.8655,13.9239,3.7315,0.9475,0.5015,0.8279
2,5.8614,127.2447,11.2803,0.6375,0.6301,0.4404
3,2.7950,12.2873,3.5053,0.8806,0.4559,0.7295
4,14.1265,766.4213,27.6843,0.3323,0.8357,0.9110


In [ ]:
final_model = finalize_model(tuned_best)
